In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import make_scorer, mean_squared_log_error
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
import shap
#import os
#os.makedirs('/root/.kaggle', exist_ok=True)


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#!mkdir -p /root/.config/kaggle
#!cp /root/.kaggle/kaggle.json /root/.config/kaggle/


# 1. Frame the Problem (15 pts)

1. Define the objective: What are you trying to predict?
   R1: The objective of this project is to predict the final sale price of a residential house property based on a set of 
   descriptive features such as physical attributes, location, and quality indicators.

2. How should you frame this problem as a machine learning problem?
   R2: This is a supervised regression problem, where the target variable (SalePrice) is continuous. The goal is to train 
   a model that learns the relationship between input features and the corresponding sale price.

3. How should performance be measured?
   R3: Model performance is evaluated using a loss function that quantifies the difference between predicted and actual sale 
   prices. In this project, the Root Mean Squared Logarithmic Error (RMSLE) is employed, as it penalizes large percentage 
   errors while being less sensitive to high-price outliers.


# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
##R1: Loading the data
train_path = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_path  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

##R2: Checking size and type

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

type(train_df), type(test_df)

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
# ==========================================
# R1: Discussion of Data
# ==========================================

## The first step is to obtain a general overview of the dataset,
## including the number of entries, feature names, and data types.
## This helps distinguish between categorical, ordinal, and numerical (int/float) variables.

train_df.info()

# ==========================================
# R2: Initial Observation of Missing Values
# ==========================================

## Many features contain missing values. For instance, Alley or PoolQC  include significantly fewer 
## entries than the expected 1460 rows. These missing values are not necessarily data collection errors —
## in several cases, NaN indicates the *absence of the feature* itself  (e.g., no alley access or no pool). 
## This will be addressed in later sections dedicated to categorical and numerical feature treatment.


In [ ]:
# ==========================================
# R3, R4, R5, R6, R7: Analysis of Categorical Features
# ==========================================

## First, all categorical features will be analyzed to identify which ones show strong influence over SalePrice.
## For this, the categorical data is isolated, and boxplots are used to visualize how SalePrice varies across categories.

# Reimport the data to ensure a clean version after previous modifications
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# --- 🧩 Identification of Categorical Features ---
cat_features = train_df.select_dtypes(include=['object']).columns
cat_features = [col for col in cat_features if col]  # safety check in case of empty column names

print("Number of categorical features:", len(cat_features))
print("Categorical features identified:", list(cat_features))

## The boxplots below represent the distribution of the target variable (SalePrice) for each category.
## They help identify features whose median or variability (box height) appears most associated with price differences.
## Taller boxes or higher median lines suggest stronger statistical correlation with SalePrice. Outliers can be identified as well.
## Uncomment the plotting section below to visualize each categorical feature’s influence.


"""
import matplotlib.pyplot as plt
import seaborn as sns

for col in cat_features:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=train_df, x=col, y='SalePrice')
    plt.title(f'SalePrice vs {col}', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
"""

## From the boxplots, some categorical features appear to have a stronger influence on house price.
## For example, ExterQual consistently shows higher median SalePrice for better exterior quality.
## Interestingly, PoolQC also presents a clear trend, but due to its imbalance (few pools in the dataset),
## including it later in the model contributes little to predictive improvement.


In [ ]:
# ==========================================
# R4, R5, R6, R7: Analysis of Numerical Features
# ==========================================

## For the numerical features, the correlation function was applied to quantify the linear relationships 
## between each feature and the target variable (SalePrice).

## This correlation analysis directly enabled the identification of the most influential numerical predictors 
## to include in the model. A light cleaning step followed to handle missing or inconsistent data in selected variables.

# --- Compute correlations among numerical features ---
corr = train_df.corr(numeric_only=True)

# --- Select the top correlated features with SalePrice ---
top_features = corr['SalePrice'].abs().sort_values(ascending=False).index[1:15]  # skip 'SalePrice' itself
print("Top 14 correlated numerical features:\n", list(top_features))

## These top correlated features represent the strongest linear dependencies with SalePrice.
## They are prioritized in the model to enhance predictive performance while maintaining interpretability.
## While OverallQual is actually ordinal, the numerical ranking allows us the keep it as numerical.
## Subsequent preprocessing steps focus on handling potential NaN values and normalizing these variables before model training.

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
# ==========================================
# TREATMENT OF NUMERICAL FEATURES
# ==========================================

## R1, R2, R3 - Part 1
## The first 12 numerical features were complete; therefore, only GarageYrBlt and MasVnrArea were revised.
## Interestingly, for the models to properly capture the trends, assigning an "old" garage year proved
## better than filling NaNs with zero, as zero could imply a non-existent or invalid construction year.

## For the numerical features selection, the correlation function was used, and the top variables
## were kept for training. This ensures that the most linearly dependent features with SalePrice are used.

## Data cleaning and feature engineering.
## The top 12 features are complete. Only the last two — GarageYrBlt and MasVnrArea — necessitated NaN treatment.


# --- Treatment of Garage Year Built ---
train_df['GarageYrBlt'] = train_df['GarageYrBlt'].fillna(1861)  # If no garage, treat as very old [baseline year: civil war:)]
test_df['GarageYrBlt']  = test_df['GarageYrBlt'].fillna(1861)

# --- Treatment of Masonry Veneer Area ---
train_df['MasVnrArea'] = train_df['MasVnrArea'].fillna(0)       # NaN means the house has no veneer → 0 m²
test_df['MasVnrArea']  = test_df['MasVnrArea'].fillna(0)


## From this point onward, including additional numerical features did not significantly improve
## the model performance; hence, the focus remained on these top predictors.

In [ ]:
# ==========================================
# TREATMENT OF CATEGORICAL / ORDINAL FEATURES
# ==========================================

## R1, R2, R3 - Part 2
## This part of the selection was more brute-force-like. From the plotting of the categorical features,
## the ones appearing to have a stronger influence on the house price were utilized.

## From the boxplot, taller boxes or higher median lines were identified; these features appear to be highly correlated
## with the SalePrice, at least from a statistical point of view; thus, the relevant ordinal/categorical features were included.

## It is interesting to see that some of the encodings in the dataset suggest NaNs; these, however, actually represent
## the lack of this feature in the building. For example, NaN in the pool means that there is no pool in the house.


# --- Treatment of Central Air ---
train_df['CentralAir'] = train_df['CentralAir'].map({'Y': 1, 'N': 0}).fillna(-1)
test_df['CentralAir']  = test_df['CentralAir'].map({'Y': 1, 'N': 0}).fillna(-1)


# --- Treatment of Pool Quality ---
pool_map = {
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    np.nan: -1  # No pool
}
train_df['PoolQC'] = train_df['PoolQC'].map(pool_map).fillna(-1)
test_df['PoolQC']  = test_df['PoolQC'].map(pool_map).fillna(-1)
## This feature ended up being irrelevant (only 10 pools → highly unbalanced).


# --- Treatment of Electrical System ---
elec_map = {
    'SBrkr': 5,
    'FuseA': 4,
    'FuseF': 3,
    'FuseP': 2,
    'Mix': 1,
    np.nan: -1
}
train_df['Electrical'] = train_df['Electrical'].map(elec_map).fillna(-1)
test_df['Electrical']  = test_df['Electrical'].map(elec_map).fillna(-1)


# --- Treatment of Fireplace Quality ---
fpq_map = {
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    'Po': 1,
    np.nan: -1  # No fireplace
}
train_df['FireplaceQu'] = train_df['FireplaceQu'].map(fpq_map).fillna(-1)
test_df['FireplaceQu']  = test_df['FireplaceQu'].map(fpq_map).fillna(-1)


# --- Treatment of Exterior Quality ---
exq_map = {
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    'Po': 1
}
train_df['ExterQual'] = train_df['ExterQual'].map(exq_map).fillna(-1)
test_df['ExterQual']  = test_df['ExterQual'].map(exq_map).fillna(-1)


# --- Treatment of Alley ---
al_map = {
    'Pave': 1,
    'Grvl': 0,
    np.nan: -1  # No Alley
}
train_df['Alley'] = train_df['Alley'].map(al_map).fillna(-1)
test_df['Alley']  = test_df['Alley'].map(al_map).fillna(-1)


# --- Treatment of Neighborhood ---
nb_map = {
    'Blmngtn': 0, 'Blueste': 1, 'BrDale': 2, 'BrkSide': 3, 'ClearCr': 4,
    'CollgCr': 5, 'Crawfor': 6, 'Edwards': 7, 'Gilbert': 8, 'IDOTRR': 9,
    'MeadowV': 10, 'Mitchel': 11, 'Names': 12, 'NoRidge': 13, 'NPkVill': 14,
    'NridgHt': 15, 'NWAmes': 16, 'OldTown': 17, 'SWISU': 18, 'Sawyer': 19,
    'SawyerW': 20, 'Somerst': 21, 'StoneBr': 22, 'Timber': 23, 'Veenker': 24,
    np.nan: -1
}
train_df['Neighborhood'] = train_df['Neighborhood'].map(nb_map).fillna(-1)
test_df['Neighborhood']  = test_df['Neighborhood'].map(nb_map).fillna(-1)
## Using fillna(-1) as a safety fallback ensures any unseen category (e.g., typos or rare neighborhoods) is handled consistently.


# --- Treatment of Condition1 and Condition2 ---
## These features describe proximity to roads, railways, parks, and other physical conditions affecting property value.
## The encoding reflects approximate desirability and correlation trends with SalePrice.

cond_map = {
    'Artery': 0,   # Adjacent to arterial street
    'Feedr': 1,    # Adjacent to feeder street
    'Norm': 2,     # Normal (typical) condition
    'RRNn': 3,     # Within 200' of North–South railroad
    'RRAn': 4,     # Adjacent to North–South railroad
    'PosN': 5,     # Near positive feature (park, greenbelt, etc.)
    'PosA': 6,     # Adjacent to positive feature
    'RRNe': 7,     # Within 200' of East–West railroad
    'RRAe': 8,     # Adjacent to East–West railroad
    np.nan: -1
}

train_df['Condition1'] = train_df['Condition1'].map(cond_map).fillna(-1)
test_df['Condition1']  = test_df['Condition1'].map(cond_map).fillna(-1)

train_df['Condition2'] = train_df['Condition2'].map(cond_map).fillna(-1)
test_df['Condition2']  = test_df['Condition2'].map(cond_map).fillna(-1)


In [ ]:
## R4 - Data normalization

selected_features = list(top_features) + ['CentralAir'] + ['PoolQC'] + ['Electrical'] + ['FireplaceQu'] + ['ExterQual'] + ['Alley'] + ['Neighborhood'] + ['Condition1'] + ['Condition2']

X = train_df[selected_features]
y = train_df['SalePrice']

# ===== Step 2. Split =====
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42) #We'll use cv later anyway.

# ===== Step 3. Normalize =====
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
# ==========================================
# R1, R2 - Model Training and Evaluation
# ==========================================

# ===== Step 4. RMSLE scorer =====
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

# ===== Step 5. Define models =====
models = {
    "Unregularized Linear": LinearRegression(),
    "Regularized Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

# ===== Step 6. Data normalization =====
# Scale data once outside the loop to prevent data leakage
X_scaled = scaler.fit_transform(X)

# ===== Step 7. Cross-validation =====
cv = KFold(n_splits=10, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    if "Tree" in name:
        X_used = X  # Tree models do not require scaling
    else:
        X_used = X_scaled  # Linear and Ridge models use scaled features

    scores = cross_val_score(model, X_used, y, cv=cv, scoring=rmsle_scorer)
    results[name] = -scores.mean()

# ===== Step 8. Summary =====
print("\n📊 Model RMSLE comparison:")
for name, score in results.items():
    print(f"{name:25s}: {score:.4f}")


In [ ]:
# ==========================================================
# R3 - SHAP Explainability Analysis: Leading-Order Variables
# ==========================================================

# Re-scale data for linear models
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# =============================
# SHAP for Linear and Ridge Models
# =============================
for name, model in models.items():
    if "Linear" in name or "Ridge" in name:
        X_used = X_scaled_df
        model.fit(X_used, y)

        # Compute SHAP values
        explainer = shap.LinearExplainer(model, X_used)
        shap_values = explainer(X_used)

        # Identify top 3 features by absolute coefficient magnitude
        top3_idx = np.abs(model.coef_).argsort()[-3:][::-1]
        top_features = X.columns[top3_idx]

        print(f"\n🔍 SHAP Summary for {name} (Top 3: {', '.join(top_features)})")
        shap.summary_plot(shap_values[:, top3_idx], X_used.iloc[:, top3_idx], feature_names=top_features)

# =============================
# SHAP for Decision Tree Model
# =============================
tree_model = models["Decision Tree"]
tree_model.fit(X, y)

# Compute feature importances for reference
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

# Prepare SHAP analysis
explainer_tree = shap.TreeExplainer(tree_model)
shap_values_tree = explainer_tree(X)

# Identify top 3 features
top3_tree = importance_df["Feature"].head(3).values
X_top3_tree = X[top3_tree]

print(f"\n🌳 SHAP Summary for Decision Tree (Top 3: {', '.join(top3_tree)})")
shap.summary_plot(shap_values_tree[:, :3], X_top3_tree, feature_names=top3_tree)


It is possible to observe clear trends in this plot: higher values of OverallQual are associated with higher sale prices, while lower quality ratings reduce them. Similarly, an increase in GrLivArea (living area) strongly correlates with higher property value. Notably, the linear models produce clustered predictions due to their limited ability to represent nonlinear relationships, whereas the decision tree yields a smoother and more continuous distribution, capturing trends more effectively.

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
# ==========================================================
# Fine-tuning and Evaluation of Ridge Regression Model
# ==========================================================

## This section performs hyperparameter optimization for the Ridge regression model
## using RandomizedSearchCV, followed by a detailed error analysis of the final tuned model.

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Parameter distribution for Ridge regularization strength ---
param_dist_ridge = {'alpha': loguniform(1e-3, 1e3)}

ridge = Ridge(random_state=42)

# --- Randomized Search for hyperparameter tuning ---
random_search_ridge = RandomizedSearchCV(
    estimator=ridge,
    param_distributions=param_dist_ridge,
    n_iter=50,
    scoring=rmsle_scorer,
    cv=10,
    random_state=42,
    n_jobs=-1
)

random_search_ridge.fit(X_scaled, y)

print("Best Ridge Parameters:", random_search_ridge.best_params_)
print(f"Best Ridge RMSLE: {-random_search_ridge.best_score_:.4f}")


# ======================
# R4 - Analysis of Error
# ======================

## This section evaluates how well the fine-tuned Ridge model predictions align with actual sale prices.

# --- Use the previously fine-tuned Ridge model ---
best_model = random_search_ridge.best_estimator_

# --- Predictions on the scaled dataset ---
y_pred = best_model.predict(X_scaled)

# --- Quantitative Error Metrics ---
rmse = np.sqrt(mean_squared_error(y, y_pred))
rmsle_val = rmsle(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("\n📉 Error Metrics for Fine-Tuned Ridge Model:")
print(f"RMSE  : {rmse:.2f}")
print(f"RMSLE : {rmsle_val:.4f}")
print(f"MAE   : {mae:.2f}")
print(f"R²    : {r2:.3f}")

# --- Visualization: Predicted vs Actual SalePrice ---
plt.figure(figsize=(6,5))
sns.scatterplot(x=y, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title("Predicted vs Actual SalePrice")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.tight_layout()
plt.show()

## Interpretation:
## The scatter plot indicates that the model systematically underpredicts the most expensive houses,
## particularly those above approximately 300k USD. This performance gap suggests that the model
## struggles to capture the nonlinear relationships or rare conditions associated with high-value properties.
## The primary cause is dataset imbalance, as only a small fraction of the samples correspond to these
## high-price houses. Consequently, the model is biased toward the more frequent mid- to low-price range.


In [ ]:
# ==========================================
# Fine-tuning: Decision Tree using GridSearchCV
# ==========================================

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

# Define parameter grid
param_grid_tree = {
    'max_depth': [5, 15, 25, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8, 16]
}

tree = DecisionTreeRegressor(random_state=42)

grid_search_tree = GridSearchCV(
    estimator=tree,
    param_grid=param_grid_tree,
    scoring=rmsle_scorer,
    cv=10,
    n_jobs=-1
)

grid_search_tree.fit(X, y)

print("Best Decision Tree Parameters:", grid_search_tree.best_params_)
print(f"Best Decision Tree RMSLE: {-grid_search_tree.best_score_:.4f}")


# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

**R1: Executive summary**

The developed model predicts residential property prices based on location, physical characteristics, and building quality indicators.

By analyzing thousands of past sales, the model accurately captures the main trends in the housing market, with only minor deviations between predicted and actual prices. Most importantly, the model enables the identification of key leading-order house characteristics on house pricing, which will be benefitial when deciding high stake investments.

**R2: Technical summary**

The project developed a supervised regression model to predict residential house prices using the Kaggle Housing dataset. Data preprocessing included handling missing values, encoding categorical and ordinal features (e.g., neighborhood, exterior quality, and central air), and normalizing numerical variables to prevent scale bias. Correlation analysis identified the top numerical predictors, including overall quality, living area, and garage year built. Three models, unregularized Linear Regression, Ridge Regression, and Decision Tree, were evaluated using 10-fold cross-validation with RMSLE scoring. 

Ridge Regression and Decision Tree achieved acceptable performance (RMSLE ≈ 0.25, R² ≈ 0.79) after hyperparameter tuning via RandomizedSearchCV and GridSearchCV. SHAP analysis confirmed that overall quality, living area, and exterior quality were the most influential predictors. 

Error analysis showed minor deviations between predicted and actual values, with underprediction mainly for high-priced houses due to dataset imbalance. Overall, the final tuned models provide robust and interpretable price predictions across the dataset.

# 9. Submit your predictions to the competition

In [ ]:
# ==========================================
# FINAL SUBMISSION GENERATION (Decision Tree)
# ==========================================

## Fine-tuned Decision Tree model submission
best_tree_model = grid_search_tree.best_estimator_
X_test_tree = test_df[X.columns]
test_predictions = best_tree_model.predict(X_test_tree)
test_predictions = np.maximum(0, test_predictions)

# Submission df
submission_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
    index_col="Id"
)

submission_df["SalePrice"] = test_predictions

# Save to CSV
submission_df.to_csv("submission.csv", index=True)

print("Submission file 'submission.csv' created successfully")
submission_df.head()


In [ ]:
%%bash

cat submission.csv | head -n 5

In [ ]:
#!kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission.csv -m "Fine-tuned Decision Tree"
